# First messy draft — SME Fintech Sales Ops Pipeline

This notebook is my first attempt. I shoved everything into one place just to see if the idea works.
Later I will clean it up into real modules.

In [ ]:
import pandas as pd
import sqlite3
import os
import random
from datetime import datetime, timedelta

# TODO: move URLs to config later
BASE_URL = "https://raw.githubusercontent.com/ikebude/CRM-Sales-Analysis/main/"

os.makedirs("data", exist_ok=True)
for f in ["sales_pipeline.csv", "sales_teams.csv", "accounts.csv", "products.csv"]:
    if not os.path.exists(f"data/{f}"):
        print(f"downloading {f}")
        df = pd.read_csv(BASE_URL + f)
        df.to_csv(f"data/{f}", index=False)

In [ ]:
# load the main pipeline data
pipeline = pd.read_csv("data/sales_pipeline.csv")
print(pipeline.head())
print(pipeline.deal_stage.value_counts())

In [ ]:
# clean up and make it feel like fintech SME sales
df = pipeline.copy()
df = df.rename(columns={
    "opportunity_id": "deal_id",
    "sales_agent": "rep",
    "close_value": "value",
    "close_date": "expected_close_date",
    "engage_date": "created_date"
})

# map CRM stages to fintech SME sales stages
stage_map = {
    "Prospecting": "Lead",
    "Engaging": "Proposal",
    "Won": "Closed Won",
    "Lost": "Closed Lost"
}
df["stage"] = df["deal_stage"].map(stage_map)
df["status"] = df["stage"].apply(lambda x: "Open" if x not in ["Closed Won", "Closed Lost"] else "Closed")

# re-label products to fintech SME products
product_map = {
    "GTX Basic": "Basic Payment Gateway",
    "GTX Plus Basic": "Plus Payment Gateway",
    "GTX Plus Pro": "Pro Payment Gateway",
    "GTXPro": "Pro Lending API",
    "MG Advanced": "Advanced Cash Flow",
    "MG Special": "SME Line of Credit",
    "GTK 500": "Enterprise FX Platform"
}
df["product"] = df["product"].map(product_map).fillna(df["product"])

# shift dates so the data looks current-ish for demo purposes
df["created_date"] = pd.to_datetime(df["created_date"])
df["expected_close_date"] = pd.to_datetime(df["expected_close_date"])
max_date = df["expected_close_date"].max()
offset = pd.Timestamp.now() - max_date - pd.Timedelta(days=30)
df["created_date"] = df["created_date"] + offset
df["expected_close_date"] = df["expected_close_date"] + offset

print(df.head())

In [ ]:
# fake activity logs because the original dataset doesn't have them
activities = []
for idx, row in df.iterrows():
    n = random.randint(2, 8)
    start = row["created_date"]
    end = row["expected_close_date"] if pd.notna(row["expected_close_date"]) else pd.Timestamp.now()
    if end < start:
        end = start + timedelta(days=1)
    for i in range(n):
        days = max(0, (end - start).days)
        d = start + timedelta(days=random.randint(0, days))
        activities.append({
            "activity_id": f"ACT_{idx}_{i}",
            "deal_id": row["deal_id"],
            "activity_type": random.choice(["Call", "Email", "Meeting"]),
            "activity_date": d.strftime("%Y-%m-%d")
        })
activities = pd.DataFrame(activities)
print(activities.head())

In [ ]:
# throw it all into SQLite
conn = sqlite3.connect("crm.db")
df.to_sql("deals", conn, if_exists="replace", index=False)
activities.to_sql("activity_logs", conn, if_exists="replace", index=False)

conn.execute("""
CREATE VIEW IF NOT EXISTS v_deal_health AS
SELECT
    d.*,
    MAX(a.activity_date) AS last_activity_date,
    CAST(JULIANDAY('now') - JULIANDAY(MAX(a.activity_date)) AS INTEGER) AS days_since_last_activity
FROM deals d
LEFT JOIN activity_logs a ON d.deal_id = a.deal_id
GROUP BY d.deal_id
""")
conn.commit()

stalled = pd.read_sql_query("SELECT * FROM v_deal_health WHERE status='Open' AND days_since_last_activity > 7", conn)
print(f"stalled deals: {len(stalled)}")
stalled.head()

In [ ]:
# dump to excel
with pd.ExcelWriter("report.xlsx", engine="openpyxl") as writer:
    stalled.to_excel(writer, sheet_name="Stalled", index=False)
    df.to_excel(writer, sheet_name="Deals", index=False)
print("report.xlsx written")

In [ ]:
# fake slack alert — real webhook later
msg = f"Daily fintech ops: {len(stalled)} stalled deals need follow-up"
with open("slack_alert.txt", "w") as f:
    f.write(msg)
print(msg)